# فصل ۱۰: حافظه و نگهداری تاریخچه (Memory & History)
## بخش ۳: هرس پیام‌ها (Message Trimming) با `trim_messages`

### چرا به هرس پیام‌ها نیاز داریم؟
اگرچه نگهداری تمام پیام‌ها برای یادآوری کامل مفید است، اما با طولانی شدن مکالمه:
1. **خطای پنجره بافتار (Context Window Overflow):** ممکن است از سقف توکن مجاز مدل عبور کنیم.
2. **کاهش دقت و گیج شدن مدل:** اطلاعات قدیمیِ نامرتبط ممکن است تمرکز مدل را از سوال فعلی منحرف کند.
3. **افزایش هزینه و تاخیر:** ارسال تاریخچه بسیار بلند در هر نوبت، مصرف توکن و زمان پاسخ را بالا می‌برد.

**راه‌حل:** استفاده از تابع `trim_messages` برای نگه داشتن تنها `k` پیام آخر (یا یک سقف مشخص از توکن‌ها) درست **پیش از ارسال به مدل**، در حالی که تاریخچه کامل همچنان در `store` باقی می‌ماند.

In [ ]:
import os
from operator import itemgetter
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# بارگذاری متغیرهای محیطی
load_dotenv()

# مقداردهی اولیه مدل
chat_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.7
)

print("✅ کتابخانه‌ها و مدل با موفقیت لود شدند!")

### ۱. ساخت پرامپت، زنجیره پایه و نگهدارنده تاریخچه (`store`)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "تو یک دستیار هوشمند و دوستانه هستی. کوتاه و دقیق پاسخ بده."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

chain = prompt | chat_model

# مخزن ذخیره تاریخچه در حافظه
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

print("✅ زنجیره پایه و store ساخته شدند!")

### ۲. تعریف هرس‌کننده (`trim_messages`) و بررسی پارامترها

| نام پارامتر | توضیحات و کاربرد |
| :--- | :--- |
| `max_tokens` | سقف مجاز تاریخچه (معنی آن وابسته به `token_counter` است؛ با `len` به معنی تعداد پیام و با `chat_model` به معنی تعداد واقعی توکن است). |
| `strategy` | `"last"` (نگه داشتن پیام‌های انتهایی/جدید) یا `"first"` (نگه داشتن پیام‌های ابتدایی). |
| `token_counter` | تابع شمارشگر (`len` برای شمارش تعداد پیام‌ها یا خود `chat_model` برای شمارش توکن واقعی). |
| `include_system` | اگر `True` باشد، پیام `SystemMessage` همیشه حفظ می‌شود (بسیار توصیه‌شده در چت‌بات‌ها). |
| `allow_partial` | اگر `False` باشد، پیام‌هایی که کامل جا نمی‌شوند به صورت ناقص برش نمی‌خورند و کامل حذف می‌شوند. |
| `start_on` | مشخص می‌کند لیست هرس‌شده باید حتماً با چه نوع پیامی شروع شود (مثلاً `"human"` تا تاریخچه با پاسخ ناقص مدل آغاز نشود). |

In [ ]:
# اینجا max_tokens=2 به همراه token_counter=len یعنی تنها ۲ پیام آخر از تاریخچه نگه داشته شود
trimmer = trim_messages(
    strategy="last",
    max_tokens=2,
    token_counter=len,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

print("✅ Trimmer با ظرفیت ۲ پیام آخر پیکربندی شد!")

### ۳. اتصال Trimmer به زنجیره با `RunnablePassthrough.assign` و `RunnableWithMessageHistory`

با استفاده از `RunnablePassthrough.assign`، مقدار `chat_history` پیش از ورود به زنجیره اصلی، از فیلتر `trimmer` عبور داده می‌شود؛ در حالی که کل پیام‌ها همچنان در `store` باقی می‌مانند.

In [ ]:
chain_with_trimming = (
    RunnablePassthrough.assign(
        chat_history=itemgetter("chat_history") | trimmer
    )
    | chain
)

with_message_history = RunnableWithMessageHistory(
    chain_with_trimming,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("✅ زنجیره با هرس خودکار و حافظه آماده است!")

### ۴. اجرای چت تعاملی با استریمینگ زنده (Streaming)

> **نکته:** برای جلوگیری از خطای لیست چندوجهی در Gemini و داشتن استریم روان متن، از `chunk.text` استفاده می‌کنیم.

**سناریوی پیشنهادی برای تست فراموشی کنترل‌شده:**
1. بگویید: `سلام. خوبی؟ من طاها هستم.`
2. بگویید: `بیا با هم یک بازی بکنیم. من نام یک تیم فوتبال رو به تو میگم، تو فقط نام یک تیم همشهری اون تیم رو بهم بگو.`
3. چند تیم بگویید: `آرسنال`، `لیورپول`، `رئال مادرید`
4. در نهایت بپرسید: `حالا بگو اسم من چی بود؟` (و ببینید چون تنها ۲ پیام آخر ارسال می‌شود، مدل اسم شما و حتی قانون بازی اولیه را فراموش کرده است!)
5. برای خروج `0` را وارد کنید.

In [ ]:
print("--- شروع چت (برای خروج 0 را بزنید) ---")

config = {"configurable": {"session_id": "1234"}}

while True:
    user_input = input("User: ")
    if user_input.strip() == "0":
        print("پایان مکالمه.")
        break
    print(f"User: {user_input}")
    
    print("AI: ", end="", flush=True)
    ai_response = ""
    for chunk in with_message_history.stream(
        {"input": user_input},
        config=config
    ):
        text = chunk.text
        ai_response += text
        print(text, end="", flush=True)
    print()

### ۵. بررسی تاریخچه کامل ذخیره‌شده در `store['1234']`

نکته بسیار مهم این است که هرس کردن پیام‌ها، آن‌ها را از دیتابیس/حافظه (`store`) **حذف نمی‌کند**! بلکه فقط هنگام ارسال به مدل محدودیت اعمال می‌کند. با اجرای سلول زیر می‌توانید تمام پیام‌ها را از ابتدا مشاهده کنید:

In [ ]:
if "1234" in store:
    print(f"تعداد کل پیام‌های ثبت‌شده در حافظه: {len(store['1234'].messages)}\n")
    for msg in store["1234"].messages:
        print(f"{msg.type}: {msg.content}")
else:
    print("هنوز پیامی برای session_id='1234' ثبت نشده است.")